# Seminar HCI and BCI in practice
## Session 3 Filtering and frequency spectrum

***In this session the prepared ECoG data are filtered and transformed to frequency space.***

Our ECoG data should be bandpass filtered in the range of [0.3 200] Hz, because most movement related information is expected to occur in the high-gamma band >65 Hz. But to understand filters a bit better, we will first have a look at some simulated data and then one single trial of the ECoG data, before actually filtering the whole ECoG data. 

In [5]:
import numpy as np
import os
import pickle

# # if you want the plots pop out from Notebook, uncomment the magic command %
%matplotlib qt
import matplotlib
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt, freqz
from src.ecog_segment_ts import ecog_segment_ts
from src.multitaper_spectrum import multitaper_spectrum
from src.signal_filter_analyzer import SignalFilterAnalyzer

main_path = os.getcwd()
main_path


'/Volumes/Data/101_PhD_Teaching/BCI_2026'

## Simulated data example

In [6]:
# Creating simulated data
Fs = 1000  # Sample frequency
L = 1000   # Length of simulated data
t = np.arange(0, L) * (1 / Fs)  # Time points
y = 0.7 * np.sin(2 * np.pi * 50 * t) + np.sin(2 * np.pi * 120 * t)  # 50 Hz and 120 Hz
np.random.seed(10)      # set seed for reproducibility
y = y + 2 * np.random.randn(len(t))  # Adding random noise

# Filter Initialization
analyzer = SignalFilterAnalyzer(data=y, fs=Fs)

### Play with the different filter parameters and methods to see how they affect the results.

To run the function `analyzer.apply_filter(order, cutoff, btype, filter_method)` properly, ensure your inputs match the following expected formats:

* **Filter Type (`btype`)**: **Accepted values:** `'lowpass'` , `'highpass'` , and `'bandpass'`.


* **Filter Order (`order`)**: An **integer** (`int`) determining the steepness of the filter. **Examples:** `2`, `4`, `8`.


* **Cutoff Frequency (`cutoff`)**: **A number** or **a list of numbers** representing the frequency limits in Hz. **Examples:* `30` (for low/high-pass) or `[30, 120]` (for band-pass).


* **Filter Method (`filter_method`)**: **Accepted values:** `'lfilter'` or `'filtfilt'`.

In [7]:
# Example: order of 8, bandpass filter between 30-120 Hz, using filtfilt
# Important: the spectrum and filter characteristics plots always based on the result of calling 'analyzer.apply_filter()' function, so if you want to see the effect of different filters, you need to call this function with different parameters before plotting the spectrum and filter characteristics.
data_ff = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='filtfilt')

# Now you can look into the effect of the filter by plotting the spectrum
analyzer.plot_spectrum(log_scale=True)

# Next move to the filter characteristics to see how the filter behaves in the frequency domain
analyzer.plot_filter_characteristics()

In [8]:
# The signal is crowed, you can plot a zoomed in version of the signal to see the effect of the filter more clearly
# By using a dict, you can compare the results of different filters, for example filtfilt vs lfilter with the same parameters
# The keys are also the labels for the legend, so you can name them as you like

data_ff = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='filtfilt')
data_lf = analyzer.apply_filter(order=8, cutoff=[30, 120], btype='bandpass', filter_method='lfilter')
compare_dict = {
    'ffilter (Order 8)': data_ff,
    'lfilter (Order 8)': data_lf
}

# Plot the both filtered data with original data together, inside the time interval of 200-300 ms
analyzer.plot_zoomed_comparison(
    compare_dict,
    time_interval=[200, 300],   # zoomed time interval
    marked_positions=[]         # here you can enter one or multiple time points, to mark them in the plot
)

---
<h2 style="color: #FF0000; font-weight: bold;">Task 1: Filter Order & Method Selection (2 pt):</h2>

Use the `SignalFilterAnalyzer` to compare different filter designs and answer the following questions.

<h3 style="color: #FF0000; font-weight: bold;">1.1 The Impact of Filter Order</h3>

Design different filters and compare **different orders** of the filters.

* How does increasing the order affect the steepness of the Gain curve (frequency domain)?

* What is the specific "cost" or negative side-effect of using a higher-order filter on the time-domain signal?

<h3 style="color: #FF0000; font-weight: bold;">1.2 `lfilter` vs. `filtfilt` in Real-World BCI</h3>

Compare the time-domain results of `lfilter` (one-way) and `filtfilt` (two-way zero-phase) using an Order 8 filter.

* **Signal Impact:** Explain the main difference between how these two methods affect the timing of the signal peaks. Give a certain time point as an example to illustrate your point.

* **Scenario:** Imagine you are building a real-time (**Online**) BCI system to control a robotic arm instantly. Which filtering method (`lfilter` or `filtfilt`) MUST you choose? Explain why based on how the algorithms access data. *(Hint: Can you see the future?)*

In [9]:
# Load data file to workspace (results from session 2)
ecog_file = os.path.join(main_path, 'data/raw/ecogStruct_Ses03.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())

dict_keys(['data', 'timebase', 'sampDur', 'nSamp', 'selectedChannels', 'srate', 'badChannels', 'badIntervals', 'refChanTS'])


In [10]:
# Load trial onset information
epoch_file = os.path.join(main_path, 'epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print(epoch.keys())

dict_keys(['OnsetIdx', 'label', 'labelOrder', 'srate', 'shift'])


In [11]:
# Cut timeseries in single trials defined by onset values in epoch2.pkl
print(f'Before processing, ecog[\'data\'] shape is {np.array(ecog['data']).shape}')
ecog = ecog_segment_ts(ecog, epoch['OnsetIdx'], 0, round(0.25 * epoch['srate']))
print(f'After processing, ecog[\'data\']shape is {ecog['data'].shape}')
print(f'epoch[\'OnsetIdx\'] shape is{np.array(epoch['OnsetIdx']).shape}')
print(f'ecog[\'timebase\'] shape is {ecog['timebase'].shape}')

Before processing, ecog['data'] shape is (40, 522868)
After processing, ecog['data']shape is (40, 254, 314)
epoch['OnsetIdx'] shape is(314,)
ecog['timebase'] shape is (254,)


---

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 (2 pt):</h2>

Have a look at the `dict` `ecog`. What has changed? How is the data arranged? 



<div style="color: #FF0000; font-weight: bold;">
    Have a look at the <code style="color: #FF0000;">`dict` `ecog`</code>. What did the function <code style="color: #FF0000;">'ecog_segment_ts'</code> done to the data?

After processing, how is the data arranged? What is the meaning of each dimension?
    </div>

<h3 style="color: #FF0000; font-weight: bold;">Your Answer to the question above:</h3>


<h3 style="color: #FF0000; font-weight: bold;">Finish the following code cell</h3>
Choose a random trial between 1 and length(epoch.label) and a random channel between 1 and 40 - try to avoid previously rejected channels!

**Note:**  [np.random.randint](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randint.html)

In [12]:
# Randomly select a channel and a trial
select_channel = ...
select_trial = ...

# Extract the data for the selected channel and trial
data = np.squeeze(ecog['data'][select_channel - 1,:, select_trial - 1])
data.shape

(254,)

In [16]:
# Nyquist frequency
nyquist_freq = 1000 / ecog['sampDur'] / 2

# Design a bandpass filter: Order 3, bandpass (0.3-200 Hz)
b, a = butter(3, [0.3 / nyquist_freq, 200 / nyquist_freq], btype='bandpass')

# Apply the filter
data_filtered = filtfilt(b, a, data)

---
<h2 style="color: #FF0000; font-weight: bold;">TASK 3 (Discussion, 1pt):</h2>

* What is the Nyquist Frequency? 

* Why do we need it here for the butterworth filter design? 

* Use the `plot_spectrum` method in the class `SignalFilterAnalyzer` to visualize the results of the filter.

*(Look at a few different trials and change the filter order to see the influence) Are you satisfied with the results of the filter?*

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration:</h3>

---
## Our ECoG Data
Filtering the whole data

In [14]:
# Reload the data
ecog_file = os.path.join(main_path, 'data/raw/ecogStruct_Ses03.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

print(ecog.keys())
len(ecog['data'])

dict_keys(['data', 'timebase', 'sampDur', 'nSamp', 'selectedChannels', 'srate', 'badChannels', 'badIntervals', 'refChanTS'])


40

In [15]:
# Nyquist frequency
nyquistFreq = 1000/ecog['sampDur']/2

# Compute filter coefficients a & b
b, a = butter(3, [0.3 / nyquist_freq, 200 / nyquist_freq], btype='bandpass')

# filtfilt operates along columns
tmp = np.array(ecog['data']).T

# Filter the data
ecog['data'] = filtfilt(b,a,tmp).T
ecog['data'].shape

(40, 522868)

---
## Cut single trials from time series 

In [32]:
# Cut timeseries in single trials defined by onset values in epoch2.pkl
ecog = ecog_segment_ts(ecog, epoch['OnsetIdx'], 0, round(0.25 * epoch['srate']))

In [33]:
ecog['data'].shape

(40, 254, 314)

---
## Frequency spectrum

To transform the data to frequency space the Spectral Analysis function `multitaper_spectrum.py` is used. The second input to this function has to be a `dict` (params) containing six specific fields (tapers, Fs, fpass, pad, err, trialave). The following Code block you have already seen in Session_02

In [36]:
# Set sepctrum analysis parameters
params = {
    'tapers': [3, 5],  # TW(time-bandwidth product)=3, K(number of tapers)=5
    'pad': 0,  # no padding
    'Fs': 1000 / ecog['sampDur'],
    'fpass': [0, 200],
    'err': 0,
    'trialavg': False
}

# Multitaper strectral analysis
ecog['data'] = np.array(ecog['data']) # As data originally saved in a list

# Initialize f and S
f, S = [],[]

n_epochs = ecog['data'].shape[2]
for epo_idx in range(n_epochs):
    ecog_oneEpo = {
        'data': ecog['data'][:,:,epo_idx],
        'sampDur': ecog['sampDur']
    }
    f_tem, S_tem = multitaper_spectrum(ecog_oneEpo, params)
    f.append(f_tem)
    S.append(S_tem)
f = np.array(f)
S = np.array(S)

# Create a new Dict to store multitaper spectral analysis results
periodogram = {
    'trailList': 1,
    'params': params,
    'periodogram':S,
    'centerFrequency':f
}

# update ecog dict
ecog['periodogram'] = periodogram

# Save the spectrual analysised data 
with open("ecogStruct1_processed.pkl", "wb") as file:
    pickle.dump(ecog, file)

# Check again your data
print(ecog.keys())
print(ecog['periodogram'].keys())
for key, value in ecog['periodogram'].items():
    print(f"Key: {key}, Type: {type(value).__name__}")

dict_keys(['data', 'timebase', 'sampDur', 'nSamp', 'selectedChannels', 'srate', 'badChannels', 'badIntervals', 'refChanTS', 'nBaselineSamp', 'periodogram'])
dict_keys(['trailList', 'params', 'periodogram', 'centerFrequency'])
Key: trailList, Type: int
Key: params, Type: dict
Key: periodogram, Type: ndarray
Key: centerFrequency, Type: ndarray


<h2 style="color: #FF0000; font-weight: bold;">TASK 4 (2 pt):</h2>

Have a look into the function `multitaper_spectrum` function.

- What does the `structure` `params` contain? 

- What is the multi-taper spectral estimation method and what are tapers? 

- What does the definition ecog.periodogram.params.pad = 1 lead to? What happens if pad is -1 or 0?

- What is padding? Why it is necessary? 

- What are the outputs `[s,f]` going to contain?

- And where will this information be stored in the `ecog` `dictionary`?
